# Canopy Water Content Calculations Using NEON Surface Reflectance Data
*July 17, 2025*

This notebook shows how to calculate equivalent water thickness (EWT) or canopy water content (CWC) from airborne hyperspectral reflectance data tiles collected at the [National Ecological Observatory Network (NEON) Soaproot Saddle (SOAP)](https://www.neonscience.org/field-sites/soap) field site. The SOAP site is in the Sierra National Forest in California and the EMIT data we use in this notebook is from July 31, 2023. This notebook is modeled after an existing notebook created by the Land Processes Distributed Active Archive Center (LP DAAC), which can be found in the NASA/VITALS GitHub repository. The existing notebook is called [3 Equivalent Water Thickness/Canopy Water Content from Imaging Spectroscopy Data](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html). Throughout the rest of the notebook below, anything in block quotes is from this existing notebook.

EMIT L2A Dataset Citation:

Green, R. (2022). <i>EMIT L2A Estimated Surface Reflectance and Uncertainty and Masks 60 m V001</i> [Data set]. NASA Land Processes Distributed Active Archive Center. https://doi.org/10.5067/EMIT/EMITL2ARFL.001 Date Accessed: 2025-06-19

From the NASA VITALS notebook:

> **Background**

> Equivalent Water Thickness (EWT) is the predicted thickness or absorption path length in centimeters (cm) of water that would be required to yield an observed spectra. In the context of vegetation, this is equivalent to canopy water content (CWC) in g/cm^2 because a cm^3 of water has a mass of 1g.

> CWC can be derived from surface reflectance spectra because they provide information about the composition of the target, including water content. Reflectance is the fraction of incoming solar radiation reflected by Earth’s surface. Different materials reflect varying proportions of radiation based upon their chemical composition and physical properties, giving materials their own unique spectral signature or fingerprint. In particular, liquid water causes characteristic absorption features to appear in the near-infrared wavelengths of the solar spectrum, which enables an estimation of its content.

> CWC correlates with vegetation type and health, as well as wildfire risk. The methods used here to calculate CWC are based on the ISOFIT python package. The Beer-Lambert physical model used to calculate CWC is described in Green et al. (2006) and Bohn et al. (2020). It uses wavelength-dependent absorption coefficients of liquid water to determine the absorption path length as a function of absorption feature depth. Of note, this model does not account for multiple scattering effects within the canopy and may result in overestimation of CWC (Bohn et al., 2020).

> More about the [EMIT mission](https://earth.jpl.nasa.gov/emit/) and [EMIT products](https://www.earthdata.nasa.gov/centers/lp-daac).

> **References**

>* Shrestha, Rupesh. 2023. Equivalent water thickness/canopy water content from hyperspectral data. Jupyter Notebook. Oak Ridge National Laboratory Distributed Active Archive Center. https://github.com/rupesh2/ewt_cwc/tree/main
>* Bohn, N., L. Guanter, T. Kuester, R. Preusker, and K. Segl. 2020. Coupled retrieval of the three phases of water from spaceborne imaging spectroscopy measurements. Remote Sensing of Environment 242:111708. https://doi.org/10.1016/j.rse.2020.111708
>* Green, R.O., T.H. Painter, D.A. Roberts, and J. Dozier. 2006. Measuring the expressed abundance of the three phases of water with an imaging spectrometer over melting snow. Water Resources Research 42:W10402. https://doi.org/10.1029/2005WR004509
>* Thompson, D.R., V. Natraj, R.O. Green, M.C. Helmlinger, B.-C. Gao, and M.L. Eastwood. 2018. Optimal estimation for imaging spectrometer atmospheric correction. Remote Sensing of Environment 216:355–373. https://doi.org/10.1016/j.rse.2018.07.003

> **Requirements** - [NASA Earthdata Account](https://urs.earthdata.nasa.gov/home)
> - *No Python setup requirements if connected to the workshop cloud instance!*
> - Local Only Set up Python Environment - See setup_instructions.md in the /setup/ folder to set up a local compatible Python environment
> - Downloaded necessary files. This is done at the end of the 01_Finding_Concurrent_Data notebook.

**Learning Objectives:**
- Calculate CWC of an ROI from NEON reflectance data
- Compare CWC of burned an unburned data

**Tutorial Outline:**

3.1 Setup
3.2 Downloading & Converting NEON Data
3.3 Calculating CWC Across an ROI (NEON Tile)
3.4 Plot Maps and Histograms

## **3.1 Setup**

In [ ]:
# Import Packages
import os, sys #modules to create and acces file paths
import math
import numpy as np #work with multi-dimensional arrays
import xarray as xr #work with labelled multi-dimenstional arrays
from osgeo import gdal #work with raster and vector geospatial data
import rasterio as rio #work with geospatial raster data
import rioxarray as rxr #work with raster arrays
from matplotlib import pyplot as plt #plotting data
import hvplot.xarray #plot multi-dimensional arrays
import hvplot.pandas #plot DataFrames/Series
import pandas as pd #work with DataFrames
import geopandas as gpd #work with geospatial shapefiles
import earthaccess #search for, download, & stream NASA earth data

from modules.emit_tools import emit_xarray #, invert_liquid_water, beer_lambert_model, get_refractive_index
# from modules.ewt_calc import calc_ewt #calculate canopy water content fxn
from scipy.optimize import least_squares #nonlinear least-squares

# from modules.test_functions import data_download_tracker, surfrfl_hvplot_image
from modules.misc_functions import surfrfl_hvplot_image
import neonutilities as nu
import h5py #work with NEON reflectance data

# Some cells may generate warnings that we can ignore; omment below lines to see the warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
pip install neonutilities

## 3.2a Read NEON Reflectance Data into xarray object

Download the NEON bidirectional surface reflectance for the burned tile with SW corner UTM coordinates of (298000, 4100000) and unburned tile with SW corner UTM coordinates of (298000, 4101000).

Data product: DP3.30006.002 are the NEON bidirectional (BRDF- and topographic- corrected) surface reflectance

In [ ]:
# If not already downloaded, you can download the NEON reflectance data locally as follows:
nu.by_tile_aop(dpid='DP3.30006.002',
               site='SOAP',
               year='2024',
               easting=[298005, 298005],
               northing=[4100005, 4101005],
               include_provisional=True,
               token=os.environ.get("NEON_TOKEN"),
               savepath='../data/input/NEON')

In [ ]:
def list_files_in_subdirectories(start_directory):
    """
    Lists all files within a specified directory and its subdirectories.

    Args:
        start_directory (str): The path to the directory to start the search from.

    Returns:
        list: A list of full paths to all files found.
    """
    all_files = []
    for root, _, files in os.walk(start_directory):
        for file in files:
            full_path = os.path.join(root, file)
            all_files.append(full_path)
    return all_files

In [ ]:
list_files_in_subdirectories('..\\data\\input\\NEON')

In [ ]:
# define filepaths to burned neon reflectance data tiles (burned and unburned h5 files)
neon_burn_h5_fp = r'../data/input/NEON/DP3.30006.002/neon-aop-provisional-products/2024/FullSite/D17/2024_SOAP_8/L3/Spectrometer/Reflectance/NEON_D17_SOAP_DP3_298000_4100000_bidirectional_reflectance.h5'
neon_unburn_h5_fp = r'../data/input/NEON/DP3.30006.002/neon-aop-provisional-products/2024/FullSite/D17/2024_SOAP_8/L3/Spectrometer/Reflectance/NEON_D17_SOAP_DP3_298000_4101000_bidirectional_reflectance.h5'

First we need to read in the AOP reflectance h5 datasets and create xarray datasets. The function `aop_h5refl2xarray` below will do this for us. Note that this function swaps the data about the y-axis so that the geographic information is consistent with the EMIT xarray datasets. 

In [ ]:
# function that reads in a NEON AOP reflectance dataset and outputs an xarray object
# this function transposes and flips the data about the y axis so y coords can be ascending, to match the EMIT data
def aop_h5refl2xarray(h5_filename):
    with h5py.File(h5_filename) as hdf5_file:
        print('Reading in ', h5_filename)
        sitename = list(hdf5_file.keys())[0]  # Adjusted to directly access the first key
        h5_refl_group = hdf5_file[sitename]['Reflectance']
        refl_dataset = h5_refl_group['Reflectance_Data']
        refl_array = refl_dataset[()].astype('float32')
        
        refl_arrayT = np.transpose(refl_array, (1, 0, 2)) # transpose so that we have y, x, wavelengths (similar to lat, lon, wavelengths)
        refl_arrayT = refl_array[::-1, :, :]  # Flip the first axis (y)
        
        refl_shape = refl_arrayT.shape
        wavelengths = h5_refl_group['Metadata']['Spectral_Data']['Wavelength'][:] #.astype('float32')
        fwhm = h5_refl_group['Metadata']['Spectral_Data']['FWHM'][:] #.astype('float32')

        # create dictionary containing metadata information
        metadata = {}
        metadata['shape'] = refl_shape

        metadata['no_data_value'] = float(
            refl_dataset.attrs['Data_Ignore_Value'])
        metadata['scale_factor'] = float(refl_dataset.attrs['Scale_Factor'])

        # Extract the scale factor & Scale the reflectance data by the scale factor - this is memory intensive though!
        # Can do it after the fact
        # scale_factor = float(refl_dataset.attrs['Scale_Factor'])
        # refl_array = refl_array.astype(float) / scale_factor

        # Extract bad band windows
        metadata['bad_band_window1'] = (
            h5_refl_group.attrs['Band_Window_1_Nanometers'])
        metadata['bad_band_window2'] = (
            h5_refl_group.attrs['Band_Window_2_Nanometers'])

        # Initialize good_wavelengths array with 1s
        good_wavelengths = np.ones_like(wavelengths) #, dtype='float32')

        # Mark wavelengths within the bad band windows as 0
        for bad_window in [metadata['bad_band_window1'], metadata['bad_band_window2']]:
            bad_indices = np.where((wavelengths >= bad_window[0]) & (wavelengths <= bad_window[1]))[0]
            good_wavelengths[bad_indices] = 0
        good_wavelengths[-10:] = 0 # the last 10 indices also tend to be noisy

        metadata['projection'] = h5_refl_group['Metadata']['Coordinate_System']['Proj4'][()].decode('utf-8')
        metadata['spatial_ref'] = h5_refl_group['Metadata']['Coordinate_System']['Coordinate_System_String'][()].decode('utf-8')
        metadata['EPSG'] = int(h5_refl_group['Metadata']
                               ['Coordinate_System']['EPSG Code'][()])
        
        map_info = str(
            h5_refl_group['Metadata']['Coordinate_System']['Map_Info'][()]).split(",")
        # extract the resolution & convert to floating decimal number
        pixel_width = float(map_info[5])
        
        pixel_height = float(map_info[6])
        # extract the upper left-hand corner coordinates from mapInfo and cast to float
        x_min = float(map_info[3]); x_min = int(x_min)
        y_max = float(map_info[4]); y_max = int(y_max)
        
        # calculate the xMax and yMin values from the dimensions
        # xMax = left edge + (# of columns * resolution)",
        x_max = x_min + (refl_shape[1]*pixel_width); x_max = int(x_max)
        # yMin = top edge - (# of rows * resolution)",
        y_min = y_max - (refl_shape[0]*pixel_height); y_min = int(y_min)

        # Calculate UTM coordinates
        x_coords = np.linspace(x_min, x_max, num=refl_shape[1]).astype(float)
        y_coordsT = np.linspace(y_min, y_max, num=refl_shape[0]).astype(float) 
        # y coords are ascending since we flipped the y in the beginning
      
        refl_xrT = xr.DataArray(refl_arrayT, dims=["y", "x", "wavelengths"], name="reflectance",
               coords={"y": ("y", y_coordsT), "x": ("x", x_coords),
                       "wavelengths": ("wavelengths", wavelengths), 
                       "fwhm": ("wavelengths", fwhm),
                       "good_wavelengths": ("wavelengths", good_wavelengths)})
        
        # Create the Transposed dataset
        dsT = xr.Dataset({"reflectance": refl_xrT})
              
        # Add metadata as attributes
        for key, value in metadata.items():
            if key not in ['shape', 'extent', 'ext_dict']:
                dsT.attrs[key] = value

        return dsT

Use the function to convert the NEON AOP hdf5 reflectance files into xarray.Datasets. This may take a ~30 seconds to a minute to run for each tile.

In [ ]:
%%time
neon_burn_ds = aop_h5refl2xarray(neon_burn_h5_fp)

In [ ]:
# display the burned tile xarray dataset
neon_burn_ds

In [ ]:
%%time
neon_unburn_ds = aop_h5refl2xarray(neon_unburn_h5_fp)

In [ ]:
# display one band (wavelength 650 nm) of the dataset for a first look
burn_band650 = neon_burn_ds.sel(wavelengths=650, method='nearest')
burn_band650.hvplot.image(x='x', y='y',
                     xlabel='UTM x',ylabel='UTM y',
                     title='NEON AOP Reflectance RGB - SOAP Burned Tile',
                     frame_width=480, frame_height=480)

In [ ]:
# display all the variable names
list(neon_burn_ds.variables.keys())

Next we will define a function that runs a few more pre-processing steps to make sure the dataset is cleaned and has the right geospatial information. These steps include:

1. scaling the reflectance data by the scale factor (NEON data are saved in an integer format, scaled by 10000, in order to save on space)

2. setting the water vapor absorption windows (defined as "bad band windows") to NaN. Similar to the EMIT data, "good_wavelengths" are provided as one of the Coordinates in the neon_burn_refl_ds xarray dataset, so we can use that information to keep only the valid wavelengths.

3. writing the CRS (coordinate reference system information)

In [ ]:
def update_neon_xr(neon_refl_ds):

    # Set fill values equal to np.nan to improve visualization
    neon_refl_ds.reflectance.data[neon_refl_ds.reflectance.data == -9999] = np.nan
    
    # Scale by the reflectance scale factor
    neon_refl_ds['reflectance'].data = ((neon_refl_ds['reflectance'].data) /
                                        (neon_refl_ds.attrs['scale_factor']))
    
    # Set "bad bands" (water vapor absorption bands and noisy bands) to NaN
    neon_refl_ds['reflectance'].data[:,:,neon_refl_ds['good_wavelengths'].data==0.0] = np.nan

    neon_refl_ds.rio.write_crs(f"epsg:{neon_refl_ds.attrs['EPSG']}", inplace=True)
    
    return neon_refl_ds

In [ ]:
%%time
neon_burn_ds = update_neon_xr(neon_burn_ds)

In [ ]:
def gamma_adjust(rgb_ds, bright=0.2, white_background=False):
    array = rgb_ds.reflectance.data
    gamma = math.log(bright)/math.log(np.nanmean(array)) # Create exponent for gamma scaling - can be adjusted by changing 0.2 
    scaled = np.power(np.nan_to_num(array,nan=1),np.nan_to_num(gamma,nan=1)).clip(0,1) # Apply scaling and clip to 0-1 range
    if white_background == True:
        scaled = np.nan_to_num(scaled, nan = 1) # Set NANs to 1 so they appear white in plots
    rgb_ds.reflectance.data = scaled
    return rgb_ds

In [ ]:
# Plot the RGB image of the burned SOAP tile
neon_burn_rgb = neon_burn_ds.sel(wavelengths=[650, 560, 470], method='nearest')
neon_burn_rgb = gamma_adjust(neon_burn_rgb,white_background=True)
neon_burn_rgb.hvplot.rgb(y='y',x='x',bands='wavelengths',
                         xlabel='UTM x',ylabel='UTM y',
                         title='NEON AOP Reflectance RGB - SOAP Burned Tile',
                         frame_width=480, frame_height=480)

In [ ]:
%%time
neon_unburn_ds = update_neon_xr(neon_unburn_ds)

In [ ]:
# Plot the RGB image of the unburned SOAP tile
neon_unburn_rgb = neon_unburn_ds.sel(wavelengths=[650, 560, 470], method='nearest')
neon_unburn_rgb = gamma_adjust(neon_unburn_rgb,white_background=True)
neon_unburn_rgb.hvplot.rgb(y='y',x='x',bands='wavelengths',
                           xlabel='UTM x',ylabel='UTM y',
                           title='NEON AOP Reflectance RGB - SOAP Unburned Tile',
                           frame_width=480, frame_height=480)

## **3.2b Save NEON reflectance data as netcdf files

Now that we have xarray datasets for the burned and unburned reflectance tiles (`neon_burn_ds` and `neon_unburn_ds`), we can save these as netcdf files, which we can then use as inputs to the calc_ewt function.

In [ ]:
# Define output filepaths for the NEON netcdf files (burned and unburned)
neon_nc_dir = r'../data/output/NEON/netcdf'

if not os.path.exists(neon_nc_dir):
    os.makedirs(neon_nc_dir)

neon_burn_nc_fp = os.path.join(neon_nc_dir,'NEON_D17_SOAP_DP3_298000_4100000_bidirectional_reflectance.nc')
neon_unburn_nc_fp = os.path.join(neon_nc_dir,'NEON_D17_SOAP_DP3_298000_4101000_bidirectional_reflectance.nc')

In [ ]:
%%time
# Export neon_burn_ds + neon_unburn_ds and save to filepath we can use in calc_ewt function
neon_burn_ds.to_netcdf(neon_burn_nc_fp)
neon_unburn_ds.to_netcdf(neon_unburn_nc_fp)

## 3.3 Calculate CWC for NEON Burned and Unburned Tiles **

> We need some lab measurements of the complex refractive index of liquid water to obtain the wavelength-dependent absorption coefficients. They are calculated by taking four times the product of Pi and the imaginary part of the refractive index, divided by wavelength. The refractive index of liquid water per wavelength is provided by the k_liquid_water_ice.csv in the data folder. We can also preview this data to get a better understanding of the information we are using. This is shown in a separate notebook.

This [k_liquid_water_ice.csv](https://github.com/nasa/VITALS/blob/main/data/k_liquid_water_ice.csv) file can be found in the [data folder in the EMIT VITALS GitHub Repo](https://github.com/nasa/VITALS/tree/main/data). We have stored it here in the `../data/k_liquid_water_ice.csv` file in order for the calc_ewt function to work as expected.

> As mentioned in the background, we use the surface reflectance to estimate CWC. The unique spectral signatures allow identification and quantification based upon the wavelength-dependent absorption coefficients of liquid water. The EMIT mission has applied similar approaches to identify dust source minerals as well as methane point source emissions. The path length of liquid water absorption can be estimated by utilizing a least squares inversion to minimize the residuals between the EMIT reflectance and the Beer-Lambert model (Green et al.,2006), which relates the wavelength-dependent absorption to the path length the photons are traveling through the material. During the inversion, the path lengths are iteratively adjusted to match the modeled spectra to the EMIT reflectance within the water absorption feature region from 850 to 1100 nm.

> This CWC calculation on the NEON reflectance data tiles (1 km x 1 km tiles, with 1 m resolution) takes on the order of 2 hours per tile, because we’re doing the calculation on a million pixels (x 426 bands), compared to ~63,000 pixels for the EMIT data. Also note that here we provide the ewt_detection_limit to increase it from the default of 0.5 in the function. We do this because there are several regions containing plants that hold significant quantities of water in this scene.

Optionally, you can check out the calc_ewt function using `help`. We have made some minor modifications to the original function, namely including an option `is_emit` to specify whether the dataset is an EMIT dataset (True) or other, eg. NEON (False). 

Now calculate CWC for our ROI, the SOAP tiles, using the NEON reflectance data. 

In [ ]:
# import `calc_ewt_neon` function - this can be moved to the top with the other imports
# this is one of two options - we can also modify the calc_ewt function to be more generic so that it works for both EMIT and NEON datasets
# for this notebook we'll use the first option, but you could also un-comment the line below to use the separate calc_ewt_neon function
# and would have to use the slightly different syntax to call that function, see commented out cell below (option 2)
# from modules.ewt_calc2 import calc_ewt_neon 

**Now run the calc_ewt_neon function on the two NEON reflectance datasets (burned and unburned)**

In [ ]:
#set output directory
neon_cwc_dir = r'../data/output/NEON/CWC/'

if not os.path.exists(neon_cwc_out_dir):
    os.makedirs(neon_cwc_out_dir)

**Calculate CWC for the burned tile using NEON reflectance data**

This takes ~ 2 hrs to run locally on Bridget's laptop and ~45 min on Openscapes, using the highest-memory option. This should only be run once! If the COG output already exists, read that in instead.

In [ ]:
%%time
neon_burn_cwc_ds = calc_ewt(
    neon_burn_nc_fp, # neon netcdf dataset filepath
    neon_cwc_out_dir,
    ewt_detection_limit=1.5,
    return_cwc=True,
    is_emit=False # new option, added so that the function works with NEON data
)

# view neon_burn_cwc_ds 
neon_burn_cwc_ds

In [ ]:
# option 2 - use the calc_ewt_neon function
# %%time
# neon_burn_cwc_ds = calc_ewt_neon(
#     neon_burn_nc_fp, # neon netcdf dataset
#     neon_cwc_out_dir,
#     ewt_detection_limit=1.5,
#     return_cwc=True
# )

# # view neon_burn_cwc_ds 
# neon_burn_cwc_ds

In [ ]:
neon_burn_cwc_cog = os.path.join(neon_cwc_dir,'NEON_D17_SOAP_DP3_298000_4100000_CWC_burned.tif')
neon_unburn_cwc_cog = os.path.join(neon_cwc_dir,'NEON_D17_SOAP_DP3_298000_4101000_CWC_unburned.tif')

In [ ]:
# Create output COG for burned tile
# Burned input netcdf tile is NEON_D17_SOAP_DP3_298000_4100000_bidirectional_reflectance.nc
neon_burn_cwc_ds.rio.to_raster(
    raster_path=neon_burn_cwc_cog, 
    driver="COG")

In [ ]:
**Calculate CWC for the unburned tile using NEON reflectance data**

This should only be run once! If the COG output already exists, read that in instead.

In [ ]:
%%time
neon_unburn_cwc_ds = calc_ewt_neon(
    neon_unburn_nc_fp, # neon netcdf dataset
    neon_cwc_out_dir,
    ewt_detection_limit=1.5,
    return_cwc=True
)

# view neon_unburn_cwc_ds 
neon_unburn_cwc_ds

In [ ]:
# Create output COG for unburned tile
# Burned input netcdf file is NEON_D17_SOAP_DP3_298000_4101000_bidirectional_reflectance.nc
neon_burn_cwc_ds.rio.to_raster(
    raster_path=neon_burn_cwc_cog, 
    driver="COG")

In [ ]:
# Once the COG is generated, you can read it in as follows:
# neon_burn_cwc_ds = rioxarray.open_rasterio(neon_burn_cwc_cog)
neon_burn_cwc_ds = rxr.open_rasterio(neon_burn_cwc_cog, band_as_variable=True)
neon_burn_cwc_ds = neon_burn_cwc_ds.rename_vars({"band_1": "cwc"})

## 3.4 Plot Maps and Histograms of NEON CWC

In [ ]:
neon_burn_cwc_ds.hvplot.image(x='x', y='y',
                              frame_height=405, frame_width=720,
                              fontscale=2,
                              cmap='jet_r',
                              clim = (0,0.5),
                              tiles='ESRI',
                              xlabel='Longitude',
                              ylabel='Latitude',
                              title='NEON CWC - SOAP Burned Tile',
                              crs='EPSG:4326')

In [ ]:
# Once the COG is generated, you can read it in as follows:
neon_unburn_cwc_ds = rxr.open_rasterio(neon_unburn_cwc_cog, band_as_variable=True)
neon_unburn_cwc_ds = neon_unburn_cwc_ds.rename_vars({"band_1": "cwc"})

In [ ]:
neon_unburn_cwc_ds.hvplot.image(x='x', y='y',
                                frame_height=405, frame_width=720,
                                fontscale=2,
                                cmap='jet_r',
                                clim = (0,0.5),
                                tiles='ESRI',
                                xlabel='Longitude',
                                ylabel='Latitude',
                                title='NEON CWC - SOAP Unburned Tile',
                                crs='EPSG:4326')

**Histograms**

In [ ]:
burned_cwc = neon_burn_cwc_ds['cwc'].data.flatten()
unburned_cwc = neon_unburn_cwc_ds['cwc'].data.flatten()

In [ ]:
plt.figure(figsize=(8, 6))
plt.hist(burned_cwc, bins=100, alpha=0.5, label='Burned CWC', color='orange')
plt.hist(unburned_cwc, bins=100, alpha=0.5, label='Unburned CWC', color='green')
plt.xlim(0, 0.75)
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Histogram of CWC from Burned and Unburned NEON Tiles at SOAP')
plt.legend()
plt.show()